In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting/data routines
import pandas as pd

# Import your CRUD module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# Hard-coded credentials for the assignment
username = "REDACTED_DATABASE_USER"
password = "REDACTED_SET_YOUR_OWN_PASSWORD"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Retrieve all records for the initial unfiltered table
df = pd.DataFrame.from_records(db.read({}))

# Drop Mongo ObjectId column so DataTable can render cleanly
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


# Load the Grazioso Salvare logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())


# Helper function to generate filter queries based on the dashboard spec
def get_filter_query(filter_type):
    if filter_type == "water":
        return {
            "breed": {
                "$regex": "Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        return {
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        return {
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        # Reset / unfiltered
        return {}


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

app.layout = html.Div([
    html.Div(id="hidden-div", style={"display": "none"}),

    html.Center([
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image.decode()),
                style={"height": "200px"}
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),
        html.B(html.H1("CJ Mavrogeorge CS-340 Dashboard"))
    ]),

    html.Hr(),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster or Individual Tracking", "value": "disaster"},
            {"label": "Reset", "value": "reset"}
        ],
        value="reset",
        inline=True
    ),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0]
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

# Update the interactive data table based on selected filter
@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    query = get_filter_query(filter_type)
    dff = pd.DataFrame.from_records(db.read(query))

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")


# Display the breeds of animals as a pie chart
@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names="breed",
                title="Preferred Animals"
            )
        )
    ]


# Highlight selected columns in the data table
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in selected_columns]


# Update the geo-location chart for the selected data entry
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Single row selection
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result externally in Codio
app.run_server(mode="external", host="0.0.0.0", port=8050, debug=False)